# DiffPool Visualizer (Toy Example)

This notebook is a **very small, self-contained tool** to help you build intuition for **DiffPool (Differentiable Pooling for Graph Neural Networks)**.

Instead of training a full GNN, we:
- Define a tiny graph by hand.
- Define a **soft assignment matrix** `S` from nodes to clusters (what DiffPool would learn).
- Show how `S` is used to **pool node features** and **coarsen the graph structure**.

The interactive visualization lets you:
- See how strongly each node belongs to each cluster.
- View the assignment matrix `S` as a heatmap.
- See the **coarsened graph** whose nodes are clusters instead of original nodes.


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

plt.rcParams["figure.figsize"] = (12, 4)

# Define a tiny toy graph: two triangles connected by a bridge edge.
G = nx.Graph()
G.add_nodes_from(range(6))
edges = [
    (0, 1), (1, 2), (2, 0),  # left triangle
    (3, 4), (4, 5), (5, 3),  # right triangle
    (2, 3),                  # bridge between communities
]
G.add_edges_from(edges)

# Simple 1D node features (e.g., group indicator with small noise)
rng = np.random.default_rng(0)
X = np.zeros((6, 1))
X[0:3, 0] = 0.2 + 0.05 * rng.normal(size=3)  # left group
X[3:6, 0] = 0.8 + 0.05 * rng.normal(size=3)  # right group

# Adjacency matrix (dense)
A = nx.to_numpy_array(G)

# Hand-crafted soft assignment matrix S: 6 nodes -> 2 clusters
# Rows: nodes 0..5, Columns: clusters 0 and 1
S = np.array([
    [0.9, 0.1],  # node 0 mostly cluster 0
    [0.8, 0.2],  # node 1 mostly cluster 0
    [0.6, 0.4],  # node 2 somewhat mixed
    [0.4, 0.6],  # node 3 somewhat mixed
    [0.2, 0.8],  # node 4 mostly cluster 1
    [0.1, 0.9],  # node 5 mostly cluster 1
])


In [ ]:
def diffpool_step(A, X, S):
    """Simulate a single DiffPool pooling step.

    A: (N, N) adjacency
    X: (N, F) node features
    S: (N, K) soft assignment matrix

    Returns:
        A_pooled: (K, K) coarsened adjacency
        X_pooled: (K, F) pooled features
    """
    X_pooled = S.T @ X          # (K, F)
    A_pooled = S.T @ A @ S      # (K, K)
    return A_pooled, X_pooled


A_pooled, X_pooled = diffpool_step(A, X, S)

# Fixed layouts for reproducible plots
pos = nx.spring_layout(G, seed=1)


def build_coarse_graph(A_pooled, threshold=0.05):
    """Build a graph whose nodes are clusters, edges weighted by A_pooled."""
    K = A_pooled.shape[0]
    Gc = nx.Graph()
    Gc.add_nodes_from(range(K))
    for i in range(K):
        for j in range(i + 1, K):
            w = A_pooled[i, j]
            if w > threshold:
                Gc.add_edge(i, j, weight=w)
    return Gc


Gc = build_coarse_graph(A_pooled)
pos_coarse = nx.spring_layout(Gc, seed=1)


In [ ]:
def plot_diffpool_visualization(cluster_idx=0):
    """Visualize original graph, assignment matrix S, and coarsened graph.

    cluster_idx: which cluster (column of S) to highlight on the original graph.
    """
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    # Panel 1: original graph with cluster-specific coloring
    ax = axes[0]
    strengths = S[:, cluster_idx]
    nx.draw(
        G,
        pos,
        node_color=strengths,
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
        with_labels=True,
        edge_color="gray",
        ax=ax,
    )
    ax.set_title(f"Original graph\nColor = soft membership in cluster {cluster_idx}")

    # Panel 2: assignment matrix S
    ax = axes[1]
    im = ax.imshow(S, cmap="viridis", vmin=0.0, vmax=1.0, aspect="auto")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Node")
    ax.set_xticks(range(S.shape[1]))
    ax.set_yticks(range(S.shape[0]))
    ax.set_title("Assignment matrix S (nodes → clusters)")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # Panel 3: coarsened graph (clusters as nodes)
    ax = axes[2]
    cluster_features = X_pooled[:, 0]  # use 1D feature
    sizes = 1000 * (cluster_features - cluster_features.min() + 0.1)
    widths = [Gc[u][v]["weight"] * 2.0 for u, v in Gc.edges()] if len(Gc.edges()) > 0 else 1.0

    nx.draw(
        Gc,
        pos_coarse,
        node_color=list(range(Gc.number_of_nodes())),
        cmap="coolwarm",
        node_size=sizes,
        with_labels=True,
        width=widths,
        edge_color="gray",
        ax=ax,
    )
    ax.set_title("Coarsened graph\nnodes = clusters")

    plt.tight_layout()
    plt.show()


interact(
    plot_diffpool_visualization,
    cluster_idx=IntSlider(min=0, max=S.shape[1] - 1, step=1, value=0),
);
